In [ ]:
import pandas as pd
import random
from IPython.display import display, Markdown

from google.colab import files
uploaded = files.upload()

df = pd.read_excel("mcdonalds_menu_besin_degerleri.xlsx")

df.head()

In [ ]:
df["Kategori"] = df["Kategori"].replace({
    "KAHVALTILIKLAR (Devam)": "KAHVALTILIKLAR",
    "İÇECEKLER (Smoothie/Meyve Suyu)": "İÇECEKLER",
})

sorted(df["Kategori"].unique())

In [ ]:
# Enerji sütunundan sadece kcal değerini al
if df["Enerji (kcal/kj)"].dtype == object:
    df["Kalori"] = (
        df["Enerji (kcal/kj)"]
        .str.split("/").str[0].str.strip()
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# Sayısal hale getirilecek sütunlar
numeric_columns = [
    "Yağ (g)", "Doymuş Yağ (g)", "Trans Yağ (g)", "Karbonhidrat (g)",
    "Şeker (g)", "Protein (g)", "Tuz (g)", "Lif (g)", "Sodyum (mg)", "Kolesterol (mg)"
]
for col in numeric_columns:
    if df[col].dtype == object:  # sadece hâlâ metinse dönüştür
        df[col] = df[col].str.replace(",", ".", regex=False).astype(float)

df.info()

In [ ]:
# 100 g satırı farklı bir isimle kayıtlı olan ürünler için eşleşme sözlüğü
REF_ESLESME = {
    "Chicken McNuggets 4'lü": "Chicken McNuggets",
    "Chicken McNuggets 6'lı": "Chicken McNuggets",
    "Çıtır Soğan 10'lu": "Çıtır Soğan",
    "Patates kızartması - Küçük Boy": "Patates Kızartması",
    "Patates Kızartması - Orta Boy": "Patates Kızartması",
    "Patates kızartması - Büyük Boy": "Patates Kızartması",
    "Patates Kızartması - Süper Boy": "Patates Kızartması",
}

# ml miktarını Tüketim Birimi'nden ayıkla (örn. "250 ml" -> 250)
df["_ml"] = df["Tüketim Birimi"].str.extract(r"(\d+)\s*ml").astype(float)
df["_is_100g"] = df["Tüketim Birimi"] == "100 g"
df["_is_100ml"] = df["_ml"] == 100
df["_is_porsiyon"] = df["Tüketim Birimi"] == "1 porsiyon"

records = []
for (kategori, urun), g in df.groupby(["Kategori", "Ürün"], sort=False):
    is_drink = g["_ml"].notna().any()

    if is_drink:
        # İçecekler: yoğunluk referansı 100 ml, gösterilecek porsiyon ise
        # 100 ml'den büyük en küçük hacim (örn. Coca-Cola'da 250 ml)
        ref_row = g[g["_is_100ml"]]
        aday = g[g["_ml"] > 100]
        serve_row = aday.loc[[aday["_ml"].idxmin()]] if not aday.empty else g[g["_ml"].notna()].loc[[g["_ml"].idxmin()]]
    else:
        ref_row = g[g["_is_100g"]]
        serve_row = g[g["_is_porsiyon"]]
        if serve_row.empty:
            # "1 porsiyon" hiç yoksa (örn. Beyaz Peynirli-Kepekli Tost),
            # 100 g satırının kendisi porsiyon kabul edilir
            serve_row = ref_row
        if ref_row.empty and urun in REF_ESLESME:
            eslesen_urun = REF_ESLESME[urun]
            ref_row = df[(df["Kategori"] == kategori) & (df["Ürün"] == eslesen_urun) & (df["_is_100g"])]

    if ref_row.empty or serve_row.empty:
        continue

    ref, serve = ref_row.iloc[0], serve_row.iloc[0]
    porsiyon_g = round(serve["Kalori"] / ref["Kalori"] * 100, 1) if ref["Kalori"] > 0 else 100.0

    records.append({
        "Kategori": kategori,
        "Ürün": urun,
        "Tüketim Birimi": serve["Tüketim Birimi"],
        "Kalori": serve["Kalori"],
        "Porsiyon (g)": porsiyon_g,
        "Protein (g)": serve["Protein (g)"],
        "Şeker (g)": serve["Şeker (g)"],
    })

df_serving = pd.DataFrame(records)
df_serving["Protein (%)"] = (df_serving["Protein (g)"] / df_serving["Porsiyon (g)"] * 100).round(1)
df_serving["Şeker (%)"] = (df_serving["Şeker (g)"] / df_serving["Porsiyon (g)"] * 100).round(1)

print("Toplam ürün:", df["Ürün"].nunique(), "| df_serving satır sayısı:", len(df_serving))
df_serving.head()

In [ ]:
top10_calories = df_serving.nlargest(10, "Kalori")[["Ürün", "Kategori", "Kalori"]]
top10_calories

In [ ]:
for category in df_serving["Kategori"].unique():
    display(Markdown(f"## {category}"))

    table = (
        df_serving[df_serving["Kategori"] == category]
        .nlargest(5, "Kalori")[["Ürün", "Kalori"]]
        .reset_index(drop=True)
    )
    display(table)

In [ ]:
shown_products = set()

for category in df_serving["Kategori"].unique():
    display(Markdown(f"## {category}"))

    temp = df_serving[df_serving["Kategori"] == category].sort_values("Kalori")
    temp = temp[~temp["Ürün"].isin(shown_products)]
    temp = temp.head(5)[["Ürün", "Kalori"]].reset_index(drop=True)

    display(temp)
    shown_products.update(temp["Ürün"])

In [ ]:
for category in df_serving["Kategori"].unique():
    display(Markdown(f"## {category}"))

    table = (
        df_serving[df_serving["Kategori"] == category]
        .sort_values("Protein (%)", ascending=False)[
            ["Ürün", "Porsiyon (g)", "Protein (g)", "Protein (%)"]
        ]
        .rename(columns={"Protein (g)": "1 Porsiyondaki Protein (g)"})
        .reset_index(drop=True)
    )
    display(table)

    display(Markdown(
        "**Not:** Protein (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının proteinden oluştuğunu göstermektedir."
    ))

In [ ]:
for category in df_serving["Kategori"].unique():
    display(Markdown(f"## {category}"))

    table = (
        df_serving[df_serving["Kategori"] == category]
        .sort_values("Şeker (%)", ascending=False)[
            ["Ürün", "Porsiyon (g)", "Şeker (g)", "Şeker (%)"]
        ]
        .rename(columns={"Şeker (g)": "1 Porsiyondaki Şeker (g)"})
        .reset_index(drop=True)
    )
    display(table)

    display(Markdown(
        "**Not:** Şeker (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının şekerden oluştuğunu göstermektedir."
    ))

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

kategori_secici = widgets.Dropdown(
    options=sorted(df_serving["Kategori"].unique()),
    description="Kategori:",
    style={"description_width": "initial"}
)

urun_secici = widgets.Dropdown(
    description="Ürün:",
    style={"description_width": "initial"}
)

buton = widgets.Button(description="Öneri Getir", button_style="success")
cikti = widgets.Output()

def kategoriyi_guncelle(change=None):
    secilen_kategori = kategori_secici.value
    urunler = sorted(
        df_serving[df_serving["Kategori"] == secilen_kategori]["Ürün"].tolist()
    )
    urun_secici.options = urunler

kategoriyi_guncelle()
kategori_secici.observe(kategoriyi_guncelle, names="value")

def oneri_getir(b):
    with cikti:
        clear_output()

        secilen = df_serving[
            (df_serving["Kategori"] == kategori_secici.value) &
            (df_serving["Ürün"] == urun_secici.value)
        ]

        if secilen.empty:
            print("❌ Ürün bulunamadı.")
            return

        secilen = secilen.iloc[0]
        kategori = secilen["Kategori"]
        kalori = secilen["Kalori"]
        protein = secilen["Protein (%)"]
        seker = secilen["Şeker (%)"]

        ayni_kategori = df_serving[
            (df_serving["Kategori"] == kategori) &
            (df_serving["Ürün"] != secilen["Ürün"])
        ]

        az_kalori = ayni_kategori[ayni_kategori["Kalori"] < kalori]
        cok_protein = ayni_kategori[ayni_kategori["Protein (%)"] > protein]
        az_seker = ayni_kategori[ayni_kategori["Şeker (%)"] < seker]

        print(f"Seçtiğiniz ürün: {secilen['Ürün']}")
        print(f"Kategori: {kategori}")

        print("\n🔥 Daha az kalorili öneri:")
        print(random.choice(az_kalori["Ürün"].tolist()) if not az_kalori.empty else "Uygun alternatif bulunamadı. Bu kategorideki en az kalorili ürün bu. Tebrikler!")

        print("\n💪 Daha yüksek protein oranına sahip öneri:")
        print(random.choice(cok_protein["Ürün"].tolist()) if not cok_protein.empty else "Uygun alternatif bulunamadı. Bu kategorideki en proteinli ürün bu. Tebrikler!")

        print("\n🍭 Daha düşük şeker oranına sahip öneri:")
        print(random.choice(az_seker["Ürün"].tolist()) if not az_seker.empty else "Uygun alternatif bulunamadı. Bu kategorideki en az şekerli ürün bu. Tebrikler!")

buton.on_click(oneri_getir)

display(kategori_secici, urun_secici, buton, cikti)